<a href="https://colab.research.google.com/github/najwaboughanmi/FactoryGuard-AI-Predictive-Maintenance/blob/main/FactoryGuard_AI_Predictive_Maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FactoryGuard AI - Predictive Maintenance System

**End-to-End Industrial AI Solution for Machine Failure Prediction**

**Author:** [Boughanmi Najwa]  
**Date:** June 2026  
**Master's in Embedded Systems**

## 1. Project Overview

This project builds a complete predictive maintenance system using machine learning on industrial sensor data. It includes EDA, modeling, and an interactive Streamlit dashboard.

## 2. Business Problem & Objectives

- Predict machine failures before they occur
- Reduce unplanned downtime in manufacturing
- Demonstrate end-to-end ML pipeline and dashboard skills

## 3. Dataset Description

We are using the public **AI4I 2020 Predictive Maintenance Dataset** (10,000 synthetic industrial records).

## 4. Exploratory Data Analysis (EDA)
In this section, we explore the industrial dataset to understand patterns, distributions, and relationships between sensor readings and machine failures.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt

print("Starting Professional EDA for FactoryGuard AI")

Starting Professional EDA for FactoryGuard AI


In [3]:
# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

print(f"Dataset Loaded Successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Dataset Loaded Successfully!
Shape: 10,000 rows × 14 columns


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
# Failure Rate Analysis
failure_rate = df['Machine failure'].mean() * 100
print(f"\n=== Machine Failure Rate: {failure_rate:.2f}% ===")
print(f"Total Failures: {df['Machine failure'].sum()}")
print(f"Normal Cases: {len(df) - df['Machine failure'].sum()}")


=== Machine Failure Rate: 3.39% ===
Total Failures: 339
Normal Cases: 9661


In [5]:
# Failure Types Distribution
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_counts = df[failure_cols].sum()

fig = px.bar(x=failure_counts.index, y=failure_counts.values,
             labels={'x': 'Failure Type', 'y': 'Count'},
             title='Distribution of Different Failure Types')
fig.show()

In [6]:
# Temperature vs Failure
fig = px.histogram(df, x='Air temperature [K]', color='Machine failure',
                   title='Air Temperature Distribution by Machine Failure',
                   barmode='overlay')
fig.show()

## 5. Data Preprocessing
In this section, we clean the data, handle missing values, encode categorical features, and prepare the dataset for modeling.

In [7]:
print("Starting Data Preprocessing for FactoryGuard AI")

Starting Data Preprocessing for FactoryGuard AI


In [8]:
# Check for missing values
print("Missing Values per Column:")
print(df.isnull().sum())

# Check data types
print("\nData Types:")
print(df.dtypes)

Missing Values per Column:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Data Types:
UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object


In [9]:
# Create a clean copy
df_clean = df.copy()

# Drop non-essential columns (UID and Product ID are identifiers)
df_clean = df_clean.drop(['UDI', 'Product ID'], axis=1)

print(f"Original Shape: {df.shape}")
print(f"Cleaned Shape: {df_clean.shape}")

Original Shape: (10000, 14)
Cleaned Shape: (10000, 12)


In [10]:
# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

# Encode 'Type' column (L, M, H)
le = LabelEncoder()
df_clean['Type'] = le.fit_transform(df_clean['Type'])

print("Categorical encoding completed")
print("Type column unique values after encoding:", df_clean['Type'].unique())

Categorical encoding completed
Type column unique values after encoding: [2 1 0]


In [11]:
# Separate features and target
X = df_clean.drop('Machine failure', axis=1)
y = df_clean['Machine failure']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print("\nData successfully split into features (X) and target (y)")

Features shape: (10000, 11)
Target shape: (10000,)

Data successfully split into features (X) and target (y)


## 6. Feature Engineering

In this section, we create new meaningful features from the industrial sensor data and analyze correlations to improve model performance.

In [12]:
print(" Starting Feature Engineering for FactoryGuard AI")

 Starting Feature Engineering for FactoryGuard AI


In [13]:
# Create new engineered features
df_eng = df_clean.copy()

# Temperature difference (important in industrial processes)
df_eng['Temp_Difference'] = df_eng['Process temperature [K]'] - df_eng['Air temperature [K]']

# Power related features
df_eng['Torque_Power'] = df_eng['Torque [Nm]'] * df_eng['Rotational speed [rpm]']
df_eng['Tool_Wear_Rate'] = df_eng['Tool wear [min]'] / (df_eng['Rotational speed [rpm]'] + 1)

print("New features created:")
print(df_eng[['Temp_Difference', 'Torque_Power', 'Tool_Wear_Rate']].head())

New features created:
   Temp_Difference  Torque_Power  Tool_Wear_Rate
0             10.5       66382.8        0.000000
1             10.5       65190.4        0.002129
2             10.4       74001.2        0.003336
3             10.4       56603.5        0.004881
4             10.5       56320.0        0.006388


In [14]:
# Correlation with target
correlation = df_eng.corr()['Machine failure'].sort_values(ascending=False)
print("\n=== Top Features Correlated with Machine Failure ===")
print(correlation.head(10))


=== Top Features Correlated with Machine Failure ===
Machine failure        1.000000
HDF                    0.575800
OSF                    0.531083
PWF                    0.522812
TWF                    0.362904
Torque [Nm]            0.191321
Torque_Power           0.176039
Tool_Wear_Rate         0.130194
Tool wear [min]        0.105448
Air temperature [K]    0.082556
Name: Machine failure, dtype: float64


In [15]:
# Interactive Correlation Heatmap
corr_matrix = df_eng.corr()

fig = px.imshow(corr_matrix,
                text_auto=True,
                aspect="auto",
                color_continuous_scale='RdBu_r',
                title='Feature Correlation Heatmap')
fig.update_layout(height=800)
fig.show()

## 7. Model Training & Evaluation

In this section, we train an XGBoost model, evaluate its performance using multiple metrics, visualize the confusion matrix, and analyze feature importance.

In [40]:
print("Starting Model Training & Evaluation")

# Strong column name cleaning for XGBoost
import re

def clean_col_names(df):
    df_cleaned = df.copy()
    new_cols = []
    for col in df_cleaned.columns:
        # Corrected regex to remove problematic chars and avoid 'nested set' warning
        new_col = re.sub(r'[[\]<>,:]', '', str(col))   # Escaping '[' and ']' properly
        new_col = new_col.replace(' ', '_')
        new_cols.append(new_col)
    df_cleaned.columns = new_cols
    return df_cleaned

# Clean features
X_train_cleaned = clean_col_names(X_train)
X_test_cleaned = clean_col_names(X_test)

print("Column names cleaned successfully!")
print("Sample columns:", X_train_cleaned.columns.tolist()[:8])

Starting Model Training & Evaluation
Column names cleaned successfully!
Sample columns: ['Type', 'Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm', 'Torque_Nm', 'Tool_wear_min', 'TWF', 'HDF']


In [32]:
from sklearn.model_selection import train_test_split

# Re-split using engineered features (in case previous split is broken)
X = df_eng.drop('Machine failure', axis=1)
y = df_eng['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_cleaned = clean_col_names(X_train)
X_test_cleaned = clean_col_names(X_test)

print(f"Training samples: {X_train_cleaned.shape[0]:,}")
print(f"Testing samples : {X_test_cleaned.shape[0]:,}")

Training samples: 8,000
Testing samples : 2,000


In [33]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Train XGBoost Model
model = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric='auc'
)

model.fit(X_train_cleaned, y_train)

print("XGBoost Model Trained Successfully!")

XGBoost Model Trained Successfully!


In [36]:
# Predictions
y_pred = model.predict(X_test_cleaned)
y_pred_proba = model.predict_proba(X_test_cleaned)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n=== Model Performance ===")
print(f"Accuracy : {accuracy:.4f}")
print(f"AUC Score: {auc:.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                labels=dict(x="Predicted Label", y="True Label"),
                title="Confusion Matrix - Machine Failure Prediction")
fig.show()


=== Model Performance ===
Accuracy : 0.9980
AUC Score: 0.9972

=== Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1932
           1       1.00      0.94      0.97        68

    accuracy                           1.00      2000
   macro avg       1.00      0.97      0.98      2000
weighted avg       1.00      1.00      1.00      2000



In [37]:
# Feature Importance
feature_importance = model.feature_importances_
features = X_train_cleaned.columns

fi_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

fig = px.bar(fi_df.head(15), x='Importance', y='Feature',
             orientation='h',
             title='Top 15 Most Important Features')
fig.show()

print("\nTop 10 Most Important Features:")
print(fi_df.head(10))


Top 10 Most Important Features:
                 Feature  Importance
7                    HDF    0.327569
9                    OSF    0.215730
8                    PWF    0.207080
6                    TWF    0.178434
3   Rotational_speed_rpm    0.034621
4              Torque_Nm    0.007336
1      Air_temperature_K    0.005055
5          Tool_wear_min    0.004891
11       Temp_Difference    0.004444
0                   Type    0.004430


## 8. Streamlit Dashboard Development

In this section, we build an interactive web dashboard using Streamlit. This is the most visible and impressive part of the project for recruiters and portfolio.

In [41]:
print("Preparing Streamlit Dashboard for FactoryGuard AI")

Preparing Streamlit Dashboard for FactoryGuard AI


In [42]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
import joblib
import numpy as np

st.set_page_config(page_title="FactoryGuard AI", layout="wide")
st.title("🏭 FactoryGuard AI - Predictive Maintenance System")
st.markdown("**End-to-End Industrial Machine Failure Prediction**")

# Load model (we will save it later)
# model = joblib.load('factoryguard_model.joblib')

st.sidebar.header("Upload Sensor Data")
uploaded_file = st.sidebar.file_uploader("Upload CSV file", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    st.write("### Preview of Uploaded Data")
    st.dataframe(df.head())

    # Add prediction logic here later
    st.success("File uploaded successfully!")

Writing app.py


In [43]:
# Save the trained model
import joblib
joblib.dump(model, 'factoryguard_model.joblib')

print("Model saved as 'factoryguard_model.joblib'")

Model saved as 'factoryguard_model.joblib'


In [44]:
# Run Streamlit in Colab (using localtunnel)
!pip install streamlit -q
!npm install -g localtunnel

# Run this cell to start the dashboard
!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 65.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://fluffy-berries-vanish.loca.lt
^C


## 9. Results & Business Impact

## 10. Conclusion & Future Work